# 02 — 驗證 Iris 線上推論（UI + Terminal）

本 notebook 為 **操作指引**，不在 Python 內呼叫推論 API。

建議流程：
1. **OpenShift AI Dashboard** 確認 `iris-classifier` 已 **Ready**
2. 從部署詳情取得推論 URL
3. 在 **Workbench Terminal** 以 `curl` 測試

| 模型 | 格式 |
|------|------|
| `iris-classifier` | scikit-learn |

**前置**：`01` 訓練完成，Dashboard 已部署 `iris-classifier`。



## 步驟 1：確認部署狀態（OpenShift AI Dashboard）

1. 登入 **OpenShift AI Dashboard** → 專案 `rhoai-quickstart`
2. 左側 **Models**（或 **Deployments**）
3. 找到 **`iris-classifier`**，Status 應為 **Ready** / **Started**

若失敗，可在 **OpenShift Console** 查看 Pod：
- Workloads → Pods → 篩選 `iris-classifier`
- 或：OpenShift Console → Pods（搜尋 `iris-classifier`）查看狀態

## 步驟 2：取得推論 URL（Dashboard）

在 `iris-classifier` 部署詳情頁：
- 複製 **Inference endpoint** / **URL**（若顯示）

若僅在叢集內存取（Workbench 同 namespace），預設 URL 為：
```
http://iris-classifier-predictor.rhoai-quickstart.svc.cluster.local:8080
```

> MLServer 監聽 **8080**；叢集內 DNS 需帶 port。

## 步驟 3：Terminal 測試（curl）

JupyterLab → **Terminal**，執行下一格。

Token 來源：`oc whoami -t`（需已 login）或 Workbench ServiceAccount token。

In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${NB_NAMESPACE:-rhoai-quickstart}"
MODEL="iris-classifier"
URL="http://${MODEL}-predictor.${NAMESPACE}.svc.cluster.local:8080"

TOKEN=$(oc whoami -t 2>/dev/null || cat /var/run/secrets/kubernetes.io/serviceaccount/token)

echo "URL: $URL"
echo "--- setosa sample ---"
curl -s -X POST "${URL}/v2/models/${MODEL}/infer" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{"inputs":[{"name":"input-0","shape":[1,4],"datatype":"FP32","data":[5.1,3.5,1.4,0.2]}]}' | python3 -m json.tool

echo "--- versicolor sample ---"
curl -s -X POST "${URL}/v2/models/${MODEL}/infer" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{"inputs":[{"name":"input-0","shape":[1,4],"datatype":"FP32","data":[6.4,3.2,4.5,1.5]}]}' | python3 -m json.tool

## 附錄：整合至應用程式（Python 參考）

正式應用請用 HTTP client 呼叫 KServe V2 API：

```python
import requests

url = "http://iris-classifier-predictor.rhoai-quickstart.svc.cluster.local:8080"
token = "<bearer-token>"
features = [5.1, 3.5, 1.4, 0.2]

resp = requests.post(
    f"{url}/v2/models/iris-classifier/infer",
    headers={"Authorization": f"Bearer {token}"},
    json={"inputs": [{"name": "input-0", "shape": [1, 4],
                       "datatype": "FP32", "data": features}]},
    verify=False,
)
prediction = resp.json()["outputs"][0]["data"][0]  # 0=setosa, 1=versicolor, 2=virginica
```

CLI 測試腳本：`inference/sklearn-iris/clients/infer.py`